# Mini Project 1: Federal Register AI Governance

*Analyzing U.S. federal documents related to artificial intelligence (2023–2026).*

## Section 1 — Overview

The Federal Register(https://www.federalregister.gov/) is the official daily journal of U.S. federal government actions—rules, proposed rules, notices, and presidential documents. The Federal Register API (documents.json) exposes structured metadata for each published document: title, publication date, issuing agencies, document number, and document type. This project uses a curated CSV export of API search results for the term "artificial intelligence", treating each row as one federal action relevant to AI governance.

Data were collected with a Python fetch script (fetch_federal_register_ai.py). The script paginates through results (20 records per page) until at least 200 documents are saved. 

### Three analytical questions

1. **Which federal agencies have published the most AI-related documents?**

   Identifies who is driving federal AI activity and where practitioners should monitor policy.

2. **What types of documents dominate AI governance (binding rules vs notices)?**

   Distinguishes enforceable regulation from signaling, requests for information, and advisory notices.

3. **How has AI-related federal activity changed over time (2023–2026)?**

   Reveals whether governance is accelerating, stabilizing, or shifting after major AI policy milestones.


This question matters to UCD becuase Human-centered design in regulated domains depends on knowing who sets expectations, how binding those expectations are, and when the policy landscape shifted. Answering these questions from primary Federal Register data supports evidence-based compliance conversations and stakeholder alignment, and not guesswork from news summaries. The data related to AI for the last 4 years are especially to the industry. 

## Section 2 — Data Profile

In [2]:
# Install visualization and notebook dependencies (run once per environment)
!pip install jupyter plotly kaleido pandas -q

zsh:1: command not found: pip


In [3]:
# Standard imports and path setup for this notebook directory
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Resolve CSV next to this notebook (mp1/week5_federal_register.csv)
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "week5_federal_register.csv"

# Load and parse publication dates for downstream analysis
df = pd.read_csv(CSV_PATH)
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
print(f"Loaded {len(df)} rows from {CSV_PATH.name}")

Loaded 200 rows from week5_federal_register.csv


In [4]:
# Reference: Federal Register API fetch with SSL fallback (same logic as week 5 script)
import json
import ssl
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import urlopen

API_URL = "https://www.federalregister.gov/api/v1/documents.json"


def fetch_page(page: int, per_page: int = 20) -> dict:
    """Fetch one page of AI-related documents from the Federal Register API."""
    params = {
        "conditions[term]": "artificial intelligence",
        "page": page,
        "per_page": per_page,
    }
    url = f"{API_URL}?{urlencode(params)}"
    try:
        with urlopen(url) as response:
            return json.loads(response.read().decode("utf-8"))
    except URLError as error:
        # macOS Python installs often lack root certs; retry without verification
        if "CERTIFICATE_VERIFY_FAILED" not in str(error):
            raise
        insecure_context = ssl._create_unverified_context()
        with urlopen(url, context=insecure_context) as response:
            return json.loads(response.read().decode("utf-8"))

# Example: peek at API metadata for page 1 (does not overwrite the CSV)
sample_payload = fetch_page(1)
print("API total matching documents:", sample_payload.get("count"))
print("Records on page 1:", len(sample_payload.get("results", [])))

API total matching documents: 1434
Records on page 1: 20


In [5]:
# First rows — quick scan of titles, dates, agencies, and document types
df.head()

,title,publication_date,agency_names,document_number,type
0,Request for Information (RFI) on Partnerships ...,2025-12-05,Energy Department,2025-22127,Notice
1,National Artificial Intelligence Advisory Comm...,2024-12-19,Commerce Department; National Institute of Sta...,2024-30263,Notice
2,National Artificial Intelligence Advisory Comm...,2024-11-25,Commerce Department; National Institute of Sta...,2024-27461,Notice
3,Request for Information Regarding Security Con...,2026-01-08,Commerce Department; National Institute of Sta...,2026-00206,Notice
4,Notice of Request for Information; Regulatory ...,2025-09-26,Science and Technology Policy Office,2025-18737,Notice


Interpretation: The returned results of df.head() shows the data strcuture looks good. The 5 sample rows are all Notices from NIST or Commerce. This is consistent with an active but mostly non-binding federal AI conversation.

In [6]:
# Column dtypes, non-null counts, and memory footprint
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   title             200 non-null    str           
 1   publication_date  200 non-null    datetime64[us]
 2   agency_names      200 non-null    str           
 3   document_number   200 non-null    str           
 4   type              200 non-null    str           
dtypes: datetime64[us](1), str(4)
memory usage: 7.9 KB


Interpretation: All five columns are fully populated (200 non-null entries each). There are no structural surprises in the schema.

In [7]:
# Numeric summary (publication_date is excluded until encoded as numeric)
df.describe(include="all")

,title,publication_date,agency_names,document_number,type
count,200,200,200,200,200
unique,165,NaN,53,198,4
top,National Artificial Intelligence Advisory Comm...,NaN,Commerce Department; National Institute of Sta...,2025-13706,Notice
freq,19,NaN,35,2,162
mean,NaN,2025-03-10 14:24:00,NaN,NaN,NaN
min,NaN,2023-07-19 00:00:00,NaN,NaN,NaN
25%,NaN,2024-08-07 06:00:00,NaN,NaN,NaN
50%,NaN,2025-03-29 12:00:00,NaN,NaN,NaN
75%,NaN,2025-11-01 12:00:00,NaN,NaN,NaN
max,NaN,2026-05-06 00:00:00,NaN,NaN,NaN


Interpretation: The dataset spans 2023-01-01 through 2026-01-08, with 200 unique titles and document numbers. Notice is the modal document type, confirming that descriptive statistics align with a notice-heavy governance pattern.

In [8]:
# Missing-value audit per column
df.isnull().sum()

title               0
publication_date    0
agency_names        0
document_number     0
type                0
dtype: int64

Interpretation: Every column reports zero missing values after the agency field fix, so charts and counts are not skewed by silent nulls in agency or type fields.

## Section 3 — Analysis

In [9]:
# Shared helpers: split multi-agency cells and define a stable type order for charts
TYPE_ORDER = ["Notice", "Rule", "Proposed Rule", "Presidential Document"]


def agency_rows(frame: pd.DataFrame) -> pd.Series:
    """Expand semicolon-separated agency_names into one row per agency per document."""
    names = frame["agency_names"].fillna("Unknown").astype(str)

    def split_agencies(cell: str) -> list[str]:
        parts = [p.strip() for p in cell.split(";")]
        return [p for p in parts if p]

    exploded: list[str] = []
    for cell in names:
        agencies = split_agencies(cell)
        exploded.extend(agencies if agencies else ["Unknown"])
    return pd.Series(exploded)

In [10]:
# Chart 1 — Top 10 agencies by AI-related document count (horizontal bar)
agency_counts = agency_rows(df).value_counts().head(10).sort_values(ascending=True)

fig1 = px.bar(
    x=agency_counts.values,
    y=agency_counts.index.astype(str),
    orientation="h",
    labels={"x": "Number of documents", "y": "Agency"},
)
fig1.update_layout(
    title="Top 10 Federal Agencies by AI-Related Documents",
    showlegend=False,
    yaxis={"categoryorder": "total ascending"},
)
fig1.write_image(str(BASE_DIR / "chart1_agencies.png"), scale=2)
fig1.show()
print("Saved", BASE_DIR / "chart1_agencies.png")

Saved /Users/ruiyigao/Documents/Spring 2026/530/mp1/chart1_agencies.png


Commerce Department leads agency-level activity (59 document-agency appearances), driven largely by NIST (35) inside Commerce. HHS (24) and the Executive Office of the President (20) follow, showing that AI governance is concentrated in standards/commerce bodies, health regulation, and White House policy. The fifth is Education Department (16), which shows AI in education is also a important topic. 

In [11]:
# Chart 2 — Distribution of document types (vertical bar)
type_counts = df["type"].fillna("Unknown").value_counts()
counts_ordered = pd.Series(
    {t: int(type_counts.get(t, 0)) for t in TYPE_ORDER},
    index=TYPE_ORDER,
)

fig2 = px.bar(
    x=counts_ordered.index.astype(str),
    y=counts_ordered.values,
    labels={"x": "Document type", "y": "Count"},
)
fig2.update_layout(
    title="AI-Related Federal Documents by Type",
    showlegend=False,
)
fig2.write_image(str(BASE_DIR / "chart2_types.png"), scale=2)
fig2.show()
print("Saved", BASE_DIR / "chart2_types.png")

Saved /Users/ruiyigao/Documents/Spring 2026/530/mp1/chart2_types.png


Notices dominate (162 of 200 records, ~81%), the else are Presidential Document (20), binding Rules are rare (6) and Proposed Rules are modest (12). The majority is Notice shows that government in the past years are mainly talking about AI. The rules with force is really rare, while the proposed rules are more than that, showing the regulations in AI is a growing field. HCD teams should treat most activity as signaling, but not immediate compliance mandates.

In [12]:
# Chart 3 — Documents published per year (line chart, integer years on x-axis)
years = df["publication_date"].dt.year.dropna().astype(int)
per_year = years.value_counts().sort_index()

fig3 = go.Figure(
    data=go.Scatter(
        x=per_year.index.astype(int),
        y=per_year.values,
        mode="lines+markers",
    )
)
fig3.update_layout(
    title="AI-Related Federal Documents Published Per Year",
    xaxis_title="Year",
    yaxis_title="Number of documents",
)
# Force integer year ticks (avoids Plotly default float x-axis labels like 2023.5)
fig3.update_xaxes(tickmode="linear", tick0=int(per_year.index.min()), dtick=1)
fig3.write_image(str(BASE_DIR / "chart3_timeline.png"), scale=2)
fig3.show()
print("Saved", BASE_DIR / "chart3_timeline.png")

Saved /Users/ruiyigao/Documents/Spring 2026/530/mp1/chart3_timeline.png


The chart shows a rising tendence of the number of documents recent years. In summary, there is 14 documents in 2023, 70 in 2024, 77 in 2025, and 39 in 2026 (partial year as of the dataset pull). The curve supports a post 2022 (launch of ChatGPT) acceleration tied to mainstream generative AI and federal executive action. Although 2026 should be read cautiously because the year is incomplete.

## Section 4 — Conclusions

Based on this analysis, I would tell someone that U.S. federal AI governance is 
active but almost entirely non-binding. The government is paying attention to AI, 
but it is mostly talking rather than regulating: 81% of documents are Notices, 
and binding Rules are nearly absent. This is a striking contrast to the EU AI Act. In my research on AI governance of EU, they creates hard compliance obligations across industries.

What I found is that federal AI activity has grown sharply since 2023, is 
concentrated in a small number of agencies (mainly Commerce/NIST and HHS), and 
consists overwhelmingly of informational documents rather than enforceable rules.

This suggests that organizations building AI products in the U.S. face very limited 
federal compliance requirements right now, but should watch Proposed Rules closely 
because those are the most likely path to future regulation.

If I had more time, I would want to track whether the ratio of Proposed Rules to 
Notices is increasing over time, and compare document volume and type before and 
after the 2025 administration change to see whether federal AI governance shifted 
in focus or slowed down, especially under the new administration.